**Destination problem:**

It is helpful to know if a train is direct to campus. However, this cannot be deduced from arrivals information. 

All arrival boards show the same destination at stations on this branch. The intention is for passengers to board the next train and change for a direct service later.

In [1]:
import sqlite3
from pathlib import Path
import pandas as pd
DATA = Path("../data")
with sqlite3.connect(DATA / "tfl_train_data.db") as conn:
    cursor = conn.cursor()
    fby = pd.read_sql_query("SELECT * FROM arrivals WHERE station='FBY' AND direction='outbound'", conn)
fby.groupby("location")["destination"].unique()

location
Approaching East Putney                                                       [Edgware Road]
Approaching Fulham Broadway                                                   [Edgware Road]
Approaching Fulham Broadway Platform 2                                        [Edgware Road]
Approaching Parsons Green                                                     [Edgware Road]
Approaching Southfields                                                       [Edgware Road]
Approaching Wimbledon                                                         [Edgware Road]
Approaching Wimbledon Park                                                    [Edgware Road]
At East Putney Platform 1                                                     [Edgware Road]
At Fulham Broadway Platform 2                                                 [Edgware Road]
At Parsons Green Platform 2                                                   [Edgware Road]
At Platform                                [Out Of Service, U

This is one example - train 055, around 9am on 1st Aug. The destination varies depending on where the train is.

In [2]:
fby[(fby["train_no"]=="055") & (fby["device_query_time"].str.startswith("2026-08-01 09"))][["device_query_time", "destination", "location"]]

,device_query_time,destination,location
412,2026-08-01 09:05:18.164767+00:00,Edgware Road,Approaching Southfields
415,2026-08-01 09:10:13.494611+00:00,Upminster,Between East Putney and Putney Bridge
419,2026-08-01 09:15:12.299457+00:00,Edgware Road,Approaching Fulham Broadway
422,2026-08-01 09:20:13.633403+00:00,Edgware Road,Approaching Fulham Broadway


A workaround is to find the set of train numbers on the line which serve stations on the direct branch and the branch requiring changes.

Data was collected on the 5th August every five minutes, polling from High Street Kensington (requiring change) and Gloucester Road (direct). In the future, the time period of collection should be extended so any patterns can be verified.

In [3]:
with sqlite3.connect(DATA / "tfl_train_data.db") as conn:
    cursor = conn.cursor()
    df = pd.read_sql_query("SELECT * FROM train_numbers", conn)
hsk_train_nos = df[df["station"]=="HSK"]["train_no"].unique()
hsk_train_nos

array(['073', '074', '075', '076', '077', '070', '071', '072'],
      dtype=object)

It can be checked if any train numbers observed at High Street Kensington were also observed at Gloucester Road.

In [4]:
df[(df["station"]=="GTR") & (df["train_no"].isin(hsk_train_nos))]

,id,train_no,station,query_time


An assumption can be made: if a train has number not in the set of numbers observed at High Street Kensington, then it is likely to be direct.